# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathibhaShaliniS/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Build the exact feature vector used in w05_model.ipynb — same engineering, same exclusions — so this leakage check is testing the real features, not a stand-in. Includes two engineered "has data" flags for the columns with structural missingness found in the data contract (search_volume, word_count), one-hot encoding for categoricals, and numeric fills.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrathibhaShaliniS/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# same restriction as w05_model.ipynb: median-CTR label only makes sense in these 3 tiers
usable_tiers = ["page_1", "striking", "page_3_5"]
d = df[df["position_tier"].isin(usable_tiers)].copy()
tier_median = d.groupby("position_tier")["ctr"].transform("median")
d["needs_ctr_review"] = (d["ctr"] < tier_median).astype(int)

# engineered "has data" flags for the structurally-missing fields found in the data contract
d["has_keyword_data"] = d["search_volume"].notna().astype(int)
d["has_word_count"] = d["word_count"].notna().astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "content_age_days", "days_since_last_update",
    "avg_position", "impressions_90d", "has_keyword_data", "has_word_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level",
    "freshness_tier", "word_count_tier", "position_tier",
]

# numeric: fill missing with 0 (the has_-flags above already record WHERE it was missing,
# so a fill here doesn't hide anything -- see section 2 for why this fill is safe)
num = d[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
# categorical: fill missing with an explicit "unknown" category, then one-hot encode
cat = d[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)

X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = d["needs_ctr_review"].to_numpy()

print("Rows:", len(d), "| Feature columns:", X.shape[1])
print("Label rate:", round(y.mean(), 3))
print(X.dtypes.value_counts())


Rows: 26360 | Feature columns: 36
Label rate: 0.496
float64    30
int64       6
Name: count, dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
raw_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "content_age_days", "days_since_last_update",
    "avg_position", "impressions_90d",
    "content_type", "main_intent", "competition_level", "freshness_tier",
    "word_count_tier", "position_tier",
]

missing_pct = d[raw_features].isna().mean().round(3).sort_values(ascending=False)
print("Missing rate per raw feature:")
print(missing_pct[missing_pct > 0])
print()

cat_cardinality = d[["content_type", "main_intent", "competition_level",
                      "freshness_tier", "word_count_tier", "position_tier"]].nunique()
print("Categorical cardinality (number of distinct values):")
print(cat_cardinality)


Missing rate per raw feature:
word_count           0.257
char_count           0.257
word_count_tier      0.257
competition_level    0.060
cpc                  0.055
search_volume        0.055
competition          0.055
main_intent          0.053
dtype: float64

Categorical cardinality (number of distinct values):
content_type         3
main_intent          4
competition_level    3
freshness_tier       4
word_count_tier      4
position_tier        3
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Test 1 — the direct label-derived column. Adding ctr itself as a feature (the column the label is literally computed from) pushes ROC-AUC from 0.801 to 0.999 — a near-perfect score is the confession, not a win. This confirms ctr must never be a feature, which is why it isn't.

Test 2 — the sibling column. clicks_90d isn't the label itself, but ctr = clicks_90d / impressions_90d, and impressions_90d is already a legitimate feature — so adding clicks_90d back in lets the model reconstruct ctr almost exactly. ROC-AUC jumps from 0.801 to 0.877, a real, meaningful leak even though it's one step removed from the label.

Test 3 — future windows. Not applicable here: the data contract already confirmed this is a single 90-day snapshot with no daily time series, so there's no later window to leak backward.

Test 4 — product flags. trend_direction, trend_pct, provider_used, model_used are confirmed absent from the feature set (see code).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

clients = d["client_id"].reset_index(drop=True)
unique_clients = clients.drop_duplicates().to_numpy()
best = None
for seed in range(200):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_mask = clients.isin(set(shuffled[:n_test])).to_numpy()
    gap = abs(y[~test_mask].mean() - y[test_mask].mean())
    if best is None or gap < best[0]:
        best = (gap, seed, test_mask)
_, SPLIT_SEED, test_mask = best
train_mask = ~test_mask

def eval_auc(features):
    pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=42))])
    pipe.fit(features[train_mask], y[train_mask])
    return roc_auc_score(y[test_mask], pipe.predict_proba(features[test_mask])[:, 1])

honest_auc = eval_auc(X)

X_with_clicks = X.copy()
X_with_clicks["clicks_90d"] = d["clicks_90d"].reset_index(drop=True)
leaky_clicks_auc = eval_auc(X_with_clicks)

X_with_ctr = X.copy()
X_with_ctr["ctr"] = d["ctr"].reset_index(drop=True)
leaky_ctr_auc = eval_auc(X_with_ctr)

print("Honest ROC-AUC (no ctr, no clicks_90d):", round(honest_auc, 3))
print("With clicks_90d added back in:", round(leaky_clicks_auc, 3))
print("With ctr added back in:", round(leaky_ctr_auc, 3))
print()
banned = {"trend_direction", "trend_pct", "provider_used", "model_used", "ctr", "clicks_90d"}
print("Product-flag / label-derived columns in the real feature set:", banned & set(X.columns))


Honest ROC-AUC (no ctr, no clicks_90d): 0.801
With clicks_90d added back in: 0.877
With ctr added back in: 0.999

Product-flag / label-derived columns in the real feature set: set()


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

ctr — the column the label is computed from (Test 1: pushes AUC to 0.999).
clicks_90d — algebraically tied to ctr via impressions_90d (Test 2: pushes AUC to 0.877).
trend_direction / trend_pct — FlyRank's own decline label, a product-derived judgment, not a raw observation.
provider_used / model_used — marked not-for-modeling in this repo's data dictionary.
engagement_rate, scroll_rate, sessions_90d, ai_traffic_pct — post-click behavior; predicting a pre-click outcome (CTR) from what happens after the click is causally backwards, even where it isn't technical leakage.
content_id, client_id — identity/grouping only, never signal for the model to learn from.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

excluded_fields = {
    "ctr": "defines the label directly (Test 1: AUC -> 0.999)",
    "clicks_90d": "algebraic sibling of ctr via impressions_90d (Test 2: AUC -> 0.877)",
    "trend_direction": "FlyRank's own decision-derived flag, not a raw observation",
    "trend_pct": "same as trend_direction",
    "provider_used": "marked not-for-modeling in the data dictionary",
    "model_used": "same as provider_used",
    "engagement_rate": "post-click behavior, causally after the outcome being predicted",
    "scroll_rate": "same as engagement_rate",
    "sessions_90d": "same as engagement_rate",
    "ai_traffic_pct": "same as engagement_rate",
    "content_id": "identity only, never a feature",
    "client_id": "grouping only, never a feature",
}
for field, reason in excluded_fields.items():
    print(f"{field:20s} — {reason}")

ctr                  — defines the label directly (Test 1: AUC -> 0.999)
clicks_90d           — algebraic sibling of ctr via impressions_90d (Test 2: AUC -> 0.877)
trend_direction      — FlyRank's own decision-derived flag, not a raw observation
trend_pct            — same as trend_direction
provider_used        — marked not-for-modeling in the data dictionary
model_used           — same as provider_used
engagement_rate      — post-click behavior, causally after the outcome being predicted
scroll_rate          — same as engagement_rate
sessions_90d         — same as engagement_rate
ai_traffic_pct       — same as engagement_rate
content_id           — identity only, never a feature
client_id            — grouping only, never a feature


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.